In [ ]:
# write scripts to plot the sourcing from the ports
# Jo Cook
# date created: 09/04/2026
# last edited: 15/04/2026

In [ ]:
import polars as pl
import json
from polylabel import polylabel
import plotly.graph_objects as go
from matplotlib import cm
from matplotlib.colors import LogNorm
import numpy as np

In [ ]:
# function to update the results dataframe, code from https://github.com/pola-rs/polars/issues/6211#issuecomment-1419250004
# to use this function, specify the column to join on, and it will search for the other columns with matching names left and right and update in place
# basically, don't put the latitude and longitude columns in the join columns bit...we need to replace them not join on them
def update(self:pl.DataFrame, df_other:pl.DataFrame,  join_columns:list[str])->pl.DataFrame:
    '''Updates dataframe with the values in df_other after joining on the join_columns'''
    
    # The columns that will be updated
    columns = [c for c in df_other.columns if c not in join_columns]
    
    df_ans = (self
        .join(df_other, how='left', on=join_columns, suffix='_NEW')      
        .with_columns(**{
            c: pl.coalesce([pl.col(c+'_NEW'), pl.col(c)]) for c in columns # <-This updates the columns
        }).select(
            pl.all().exclude('^.*_NEW$') # <- this drops the temporary '*_NEW' columns
           )
       )    
    return df_ans

In [ ]:
#read in results data
results=pl.read_parquet("Y:/Academic/PostDoc/SCPGroup/Trase/DietTrase/Data/Results27thJan2026/diet-trase-results-2020.parquet")

#get the relevant columns so I don't just have the whole dataset
results=results.select(["country_of_production","mass_tonnes_raw_equivalent", "port_of_export_name", "production_geocode_id", "production_geocode_name"])

# remove subnational regions with unknown production geocodes as we can't map them
results=results.filter((pl.col("production_geocode_name")!="UNKNOWN"))

#add placeholder columns in the results for the later joins
results=results.with_columns(
    port_lon_degE=pl.lit(0.0),
    port_lat_degN=pl.lit(0.0),
    centre_lon_degE=pl.lit(0.0),
    centre_lat_degN=pl.lit(0.0)
    
)

In [ ]:
# get the port and jurisdiction data
port_jurisdiction_countries=["india", "brazil", "indonesia", "tanzania", "colombia", "peru"]

# read in and clean the ports and jurisdictions data
# and merge the jurisdictions with the results
for country in port_jurisdiction_countries:
    
    #read in the port data
    temp_ports=pl.read_parquet("Y:/Academic/PostDoc/SCPGroup/Trase/DietTrase/Data/SpatialData&Ports/"+country+"_ports_coffee_2020.parquet")
    
    #keep only the cleaned column
    temp_ports=temp_ports.select(["port_of_export_clean", "lon", "lat"])

    #rename the latitude and longitude columns to be relevant to the ports, this will help avoid confusion later
    temp_ports=temp_ports.rename({"port_of_export_clean": "port_of_export_name", "lon": "port_lon_degE", "lat": "port_lat_degN"})
    
    #cast the coordinates to floats, in some cases they are strings
    temp_ports=temp_ports.with_columns(
        pl.col("port_lon_degE").cast(pl.Float64),
        pl.col("port_lat_degN").cast(pl.Float64)
    )
    
    #merge in the ports
    results=update(results,temp_ports,['port_of_export_name'])
    
    #--------------------------
    
    #read justidictions data
    temp_jurisdictions=pl.read_parquet("Y:/Academic/PostDoc/SCPGroup/Trase/DietTrase/Data/SpatialData&Ports/diet_trase_"+country+"_coffee_2020_jurisdictions.parquet")
    
    #rename the lon lat columns to avoid confusion later
    temp_jurisdictions=temp_jurisdictions.rename({"geocode": 'production_geocode_id',"name": 'production_geocode_name', "lon": "centre_lon_degE", "lat": "centre_lat_degN"})
    
    # merge in the centre of the state/ subnational region
    results=update(results, temp_jurisdictions, ['production_geocode_id','production_geocode_name'])


In [ ]:
#get the spatial data and co-ordinates which will be used to produce the map
spatial_countries=["IND", "BRA", "IDN", "TZA"]
spatial_data=[]

for country in spatial_countries:
    with open("Y:/Academic/PostDoc/SCPGroup/Trase/DietTrase/Data/SpatialData&Ports/gadm41_"+country+"_1.json") as f:
        spatial_data.append(json.load(f))
    #coordinates are also recorded as [lon, lat]
    
# read in colmobia spatial data
with open("Y:/Academic/PostDoc/SCPGroup/Trase/DietTrase/Data/SpatialData&Ports/Municipios_DANE_USAID.geojson") as f:
        colombia_spatial=json.load(f)

#read in peru spatial data
with open("Y:/Academic/PostDoc/SCPGroup/Trase/DietTrase/Data/SpatialData&Ports/provincias.geojson") as f:
        peru_spatial=json.load(f)


Data missing so far:
- json file for Peru and Colombia that has the coordinates of the subnational districts (similar to the India , Brazil, Indonesia and Tanzania gadm level 1 files)
- update: now have these files as geojsons, so leave for now and do the rest that should have the saem format as india

Prepare data for map plotting

In [ ]:
# TODO: maybe figure a way around this, but for now I did add zeroes for the centre points and ports so as long as all are not zero we can plot for now
results=results.filter(((pl.col("centre_lon_degE")!=0.0) | (pl.col("centre_lat_degN")!=0.0)) &
                       ((pl.col("port_lon_degE")!=0.0) | (pl.col("port_lon_degE")!=0.0)))

In [ ]:
#aggregate total mass for each state-port flow and build a log-scaled colour mapping for plotting the map
flow_totals = (
    results
    .group_by([
        "production_geocode_name",
        "centre_lon_degE",
        "centre_lat_degN",
        "port_of_export_name",
        "port_lon_degE",
        "port_lat_degN"
    ])
    .agg(pl.col("mass_tonnes_raw_equivalent").sum().alias("total_mass"))
)

#make a log colour mapping for the mass, annoyingly seems to depend on pandas??? - might be because this is now bodged together from several AI helpers
flow_df = flow_totals.to_pandas()
flow_df["total_mass"] = flow_df["total_mass"].replace(0, 1e-1)
norm = LogNorm(vmin=flow_df["total_mass"].min(), vmax=flow_df["total_mass"].max())
cmap = cm.viridis
flow_df["color"] = flow_df["total_mass"].apply(
    lambda v: cmap(norm(v))
)

# create log scale for mapping
flow_df["log_mass"] = np.log10(flow_df["total_mass"])

In [ ]:
# get coordinates of the unique origins so we can add them to the map later
unique_origins = flow_df[["centre_lon_degE", "centre_lat_degN"]].drop_duplicates()

# get coordinatse of the unique ports to add to the map later
unique_ports = flow_totals.select(["port_of_export_name", "port_lon_degE", "port_lat_degN"]).unique()

In [ ]:
# initialise figure
fig = go.Figure()

fig.add_trace(go.Scattergeo(
    lon=[None],
    lat=[None],
    mode='markers',
    marker=dict(
        colorscale='Viridis',
        cmin=flow_df["log_mass"].min(),
        cmax=flow_df["log_mass"].max(),
        color=[flow_df["log_mass"].min(), flow_df["log_mass"].max()],
        colorbar=dict(title="log10(Total mass)"),
        showscale=True
    ),
    showlegend=False
))

# Plot flow lines
for _, row in flow_df.iterrows():
    fig.add_trace(go.Scattergeo(
        lon=[row["centre_lon_degE"], row["port_lon_degE"]],
        lat=[row["centre_lat_degN"], row["port_lat_degN"]],
        mode='lines',
        line=dict(width=3, color=f'rgba({int(row["color"][0]*255)},{int(row["color"][1]*255)},{int(row["color"][2]*255)},{row["color"][3]})'),
        opacity=0.7
    ))

# Origins
fig.add_trace(go.Scattergeo(
    lon=unique_origins["centre_lon_degE"],
    lat=unique_origins["centre_lat_degN"],
    mode='markers',
    marker=dict(color='blue', size=6),
    name='Origins'
))

# Ports
fig.add_trace(go.Scattergeo(
    lon=unique_ports["port_lon_degE"],
    lat=unique_ports["port_lat_degN"],
    mode='markers',
    marker=dict(color='red', size=6, symbol='x'),
    name='Ports'
))

fig.update_layout(
    geo=dict(
        scope='world', #asia
        showcountries=True,      # Show country boundaries
        countrycolor="Black",
        projection_type='natural earth',
        showland=True
    ),
    height=800,
    
    showlegend=False
)

# overlay borders onto map manually using the files from trase s3
for i in range(0,4):
    fig.add_trace(go.Choropleth(
        geojson=spatial_data[i], 
        locations=[f["properties"]["NAME_1"] for f in spatial_data[i]["features"]],
        z=[1]*len(spatial_data[i]["features"]),
        featureidkey="properties.NAME_1",  
        colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
        marker_line_color="grey",
        marker_line_width=1,
        showscale=False
    ))
    
#manually add colombia departments, structure is not the same as the above 4 countries
fig.add_trace(go.Choropleth(
        geojson=colombia_spatial, 
        locations=[f["properties"]["DPTO_CNMBR"] for f in colombia_spatial["features"]],
        z=[1]*len(colombia_spatial["features"]),
        featureidkey="properties.DPTO_CNMBR",  
        colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
        marker_line_color="grey",
        marker_line_width=1,
        showscale=False
    ))

#manually add peru's provinces, structure of spatial file is different than the other countries
fig.add_trace(go.Choropleth(
        geojson=peru_spatial, 
        locations=[f["properties"]["PROVINCIA"] for f in peru_spatial["features"]],
        z=[1]*len(peru_spatial["features"]),
        featureidkey="properties.PROVINCIA",  
        colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
        marker_line_color="grey",
        marker_line_width=1,
        showscale=False
    ))

fig.show()